In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [4]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [5]:
message.content[0].text

'Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, allowing certain problems to be solved exponentially faster.'

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [7]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [8]:
final_answer

'Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in a superposition of both states simultaneously, allowing quantum computers to explore multiple solutions in parallel.'

In [10]:
message = []

while True:
    user_input = input("You: ")
    print(f"You: {user_input}")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the chat.")
        break

    add_user_message(message, user_input)
    response = chat(message)
    add_assistant_message(message, response)

    print(f"Claude: {response}")

You: My name is MK
Claude: Nice to meet you, MK! How can I help you today?
You: What model am I talking to?
Claude: You're talking to Claude, made by Anthropic. I'm an AI assistant designed to help with a wide variety of tasks like answering questions, writing, analysis, math, coding, creative projects, and much more.

Is there something I can help you with?
You: Wich of the 3 models in claude?
Claude: I'm Claude 3.5 Sonnet, which is the most recent model in the Claude 3 family.

The Claude 3 family includes three models:
- **Claude 3 Opus** - the most capable model
- **Claude 3 Sonnet** - balanced performance and speed
- **Claude 3 Haiku** - the fastest and most compact

However, I should note that I'm actually Claude 3.5 Sonnet, which is an updated version that sits between the original Sonnet and Opus in capability, with improved performance across many tasks.
You: I thought i was talking to claude-haiku-4-5-20251001
Claude: You're right to correct me! If that's the model identifier

In [20]:
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [15]:
# Without system prompt
chat(messages)


'Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in a superposition of both states simultaneously, allowing quantum computers to explore multiple solutions in parallel.'

In [16]:
# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
chat(messages, system=system)

'Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are either 0 or 1, allowing them to explore many possible solutions to a problem in parallel.'

#### Streaming

In [18]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# FakeDB

A lightweight in-memory database that generates realistic mock data on-the-fly, allowing developers to prototype applications and run tests without needing actual data or external database connections.

#### stop sequences

In [22]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

chat(messages, stop_sequences=["```"])

'\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'